In [1]:
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import os
import ast
from datetime import datetime

# Define the directory path
directory_path = r"C:\Users\Jack\Desktop\foodRecipeAndInteractions"

for dirname, _, filenames in os.walk(directory_path):
    for filename in filenames:
        full_path = os.path.join(dirname, filename)
        print(full_path)

C:\Users\Jack\Desktop\foodRecipeAndInteractions\ingr_map.pkl
C:\Users\Jack\Desktop\foodRecipeAndInteractions\interactions_test.csv
C:\Users\Jack\Desktop\foodRecipeAndInteractions\interactions_train.csv
C:\Users\Jack\Desktop\foodRecipeAndInteractions\interactions_validation.csv
C:\Users\Jack\Desktop\foodRecipeAndInteractions\PP_recipes.csv
C:\Users\Jack\Desktop\foodRecipeAndInteractions\PP_users.csv
C:\Users\Jack\Desktop\foodRecipeAndInteractions\RAW_interactions.csv
C:\Users\Jack\Desktop\foodRecipeAndInteractions\RAW_recipes.csv
C:\Users\Jack\Desktop\foodRecipeAndInteractions\RAW_recipes_with_amount.csv


##Data Cleaning part##

In [2]:
#file_path = "C:\\Users\\Jack\\Desktop\\foodRecipeAndInteractions\\RAW_recipes.csv"
file_path = "C:\\Users\\Jack\\Desktop\\foodRecipeAndInteractions\\RAW_recipes_with_amount.csv"

recipes_df = pd.read_csv(file_path)
print(recipes_df.head(2))

                                         name      id  minutes  \
0  arriba   baked winter squash mexican style  137739       55   
1            a bit different  breakfast pizza   31490       30   

   contributor_id   submitted  \
0           47892  2005-09-16   
1           26278  2002-06-17   

                                                tags  \
0  ['60-minutes-or-less', 'time-to-make', 'course...   
1  ['30-minutes-or-less', 'time-to-make', 'course...   

                                   nutrition  n_steps  \
0      [51.5, 0.0, 13.0, 0.0, 2.0, 0.0, 4.0]       11   
1  [173.4, 18.0, 0.0, 17.0, 22.0, 35.0, 1.0]        9   

                                               steps  \
0  ['make a choice and proceed with recipe', 'dep...   
1  ['preheat oven to 425 degrees f', 'press dough...   

                                         description  \
0  autumn is my favorite time of year to cook! th...   
1  this recipe calls for the crust to be prebaked...   

                      

In [3]:
# Function to clean and preprocess the data
def preprocess_data(df):
    # Convert 'submitted' column to datetime format
    df['submitted'] = pd.to_datetime(df['submitted'], format='%d/%m/%Y', errors='coerce')
    
    # Convert 'tags' column from string representation of list to actual list
    df['tags'] = df['tags'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)
    
    # Convert 'nutrition' column from string representation of list to actual list
    df['nutrition'] = df['nutrition'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)
    
    # Convert 'steps' column from string representation of list to actual list
    df['steps'] = df['steps'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)
    
    # Convert 'ingredients' column from string representation of list to actual list
    df['ingredients'] = df['ingredients'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)
    
    # Fill missing values in 'description' with empty string
    df['description'] = df['description'].fillna('')
    
    # Fill missing values in 'steps' with empty lists
    df['steps'] = df['steps'].apply(lambda x: x if isinstance(x, list) else [])
    
    # Fill missing values in 'ingredients' with empty lists
    df['ingredients'] = df['ingredients'].apply(lambda x: x if isinstance(x, list) else [])
    
    # Remove any leading or trailing whitespace from text columns
    df['name'] = df['name'].str.strip()
    df['description'] = df['description'].str.strip()

    # Data cleaning for 'amount' column
    if 'amount' in df.columns:
        # Fill missing values in 'amount' with a default value, e.g., 0
        df['amount'] = df['amount'].fillna(0)
        
        # Ensure 'amount' is of type integer
        df['amount'] = df['amount'].astype(int)
    
    return df

In [4]:
# Apply preprocessing
recipes_df = preprocess_data(recipes_df)

# Display cleaned data
print("Data After Cleaning:")
print(recipes_df.head())

# Display specific details from the first row as an example
print("\nExample Recipe Data:")
print("Name:", recipes_df.loc[0, 'name'])
print("ID:", recipes_df.loc[0, 'id'])
print("Minutes:", recipes_df.loc[0, 'minutes'])
print("Contributor ID:", recipes_df.loc[0, 'contributor_id'])
print("Submitted Date:", recipes_df.loc[0, 'submitted'])
print("Tags:", recipes_df.loc[0, 'tags'])
print("Nutrition:", recipes_df.loc[0, 'nutrition'])
print("Number of Steps:", recipes_df.loc[0, 'n_steps'])
print("Steps:", recipes_df.loc[0, 'steps'])
print("Description:", recipes_df.loc[0, 'description'])
print("Ingredients:", recipes_df.loc[0, 'ingredients'])
print("Number of Ingredients:", recipes_df.loc[0, 'n_ingredients'])
print("Amount:", recipes_df.loc[0, 'amount'])  # Adding the amount field


Data After Cleaning:
                                         name      id  minutes  \
0  arriba   baked winter squash mexican style  137739       55   
1            a bit different  breakfast pizza   31490       30   
2                   all in the kitchen  chili  112140      130   
3                          alouette  potatoes   59389       45   
4          amish  tomato ketchup  for canning   44061      190   

   contributor_id submitted  \
0           47892       NaT   
1           26278       NaT   
2          196586       NaT   
3           68585       NaT   
4           41706       NaT   

                                                tags  \
0  [60-minutes-or-less, time-to-make, course, mai...   
1  [30-minutes-or-less, time-to-make, course, mai...   
2  [time-to-make, course, preparation, main-dish,...   
3  [60-minutes-or-less, time-to-make, course, mai...   
4  [weeknight, time-to-make, course, main-ingredi...   

                                    nutrition  n_steps  \


##Model Initialization part##

In [5]:
#pip install sentence-transformers

In [6]:
import pandas as pd
import numpy as np
import torch
from sentence_transformers import SentenceTransformer

# Initialize the SentenceTransformer model
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

# Combine 'description' and 'steps' into a single text column
recipes_df['full_description'] = recipes_df['name'] + ' ' + recipes_df['description']

# Encode recipe descriptions into embeddings
recipe_descriptions = recipes_df['full_description'].tolist()
recipe_embeddings = model.encode(recipe_descriptions, convert_to_tensor=True, device='cuda')

# Save embeddings and DataFrame to file (optional)
torch.save(recipe_embeddings, 'recipe_embeddings.pt')
recipes_df.to_pickle('recipes_df.pkl')


c:\Users\Jack\anaconda3\Lib\site-packages\sentence_transformers\cross_encoder\CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


##Similarity Function part##

In [9]:
def find_similar_recipes(prompt, dietary_restrictions='', eating_habits='', budget=''):
    # Encode the prompt using the model
    prompt_embedding = model.encode(prompt, convert_to_tensor=True, device='cuda')
    
    # Calculate cosine similarities between the prompt and recipe embeddings
    similarities = torch.nn.functional.cosine_similarity(prompt_embedding, recipe_embeddings)
    
    # Get indices of top 5 most similar recipes
    top_indices = similarities.argsort(descending=True).cpu().numpy()[:5]
    
    filtered_recipes = []
    for idx in top_indices:
        recipe = recipes_df.iloc[idx]
        
        # Filter based on dietary restrictions
        if dietary_restrictions:
            restrictions = dietary_restrictions.split(',')
            if any(restriction in recipe['ingredients'] for restriction in restrictions):
                continue
        
        # Filter based on eating habits
        if eating_habits:
            habits = eating_habits.split(',')
            if not any(habit in recipe['tags'] for habit in habits):
                continue
        
        # Filter based on budget
        if budget:
            budget = int(budget)
            if not (budget - 20 <= recipe['amount'] <= budget + 20):
                continue
        
        filtered_recipes.append(recipe)
    
    return filtered_recipes

##Flask part## 

In [10]:
from flask import Flask, render_template, request, jsonify
import random

app = Flask(__name__, template_folder='../ui')

@app.route('/', methods=['GET', 'POST'])
def home():
    if request.method == 'POST':
        prompt = request.form['prompt']
        dietary_restrictions = request.form.get('dietary_restrictions', '')
        eating_habits = request.form.get('eating_habits', '')
        budget = request.form.get('budget', '')
        
        # Find similar recipes based on user input
        similar_recipes = find_similar_recipes(prompt, dietary_restrictions, eating_habits, budget)
        
        # Prepare response with recipe names and summaries
        recipes_menu = []
        for recipe in similar_recipes:
            recipe_name = recipe['name']
            recipe_description = recipe['description']
            recipe_summary = f"{recipe_name}: {recipe_description[:100]}..."  # Example: Limit description length
            recipes_menu.append(recipe_summary)
        
        return render_template('restaurantMenuGenerator.html', prompt=prompt, menu=recipes_menu)
    
    return render_template('restaurantMenuGenerator.html', prompt='', menu='')

@app.route('/generate_random_recipe', methods=['GET'])
def random_recipe():
    random_index = random.randint(0, len(recipes_df) - 1)
    random_recipe = recipes_df.iloc[random_index]
    recipe_name = random_recipe['name']
    recipe_description = random_recipe['description']
    recipe_summary = f"{recipe_name}: {recipe_description[:100]}..."  # Example: Limit description length
    return jsonify(recipe_summary)

if __name__ == "__main__":
    app.run(debug=True)

 * Serving Flask app '__main__'
 * Debug mode: on


 * Running on http://127.0.0.1:5000
Press CTRL+C to quit
 * Restarting with watchdog (windowsapi)


SystemExit: 1